# 08 — Validation externe (Quiver)

**Rôle :** vérifier notre table contre Quiver (couverture par an / déclarant). **Quiver n'entre jamais dans la table finale** — c'est une sanity-check, pas une source.

**Prérequis :** `QUIVER_API_TOKEN` dans l'environnement · notebook 07 exécuté.

**Cellule 1 — Imports, chemins, token.**

In [ ]:
import os, json
from pathlib import Path
import requests, pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED    = PROJECT_ROOT / "data" / "processed"
EXT_QUIVER   = PROJECT_ROOT / "data" / "external" / "quiver"; EXT_QUIVER.mkdir(parents=True, exist_ok=True)
REPORTS      = PROJECT_ROOT / "reports"
TOKEN = os.environ.get("QUIVER_API_TOKEN", "").strip()
print("Token présent :", bool(TOKEN))

**Cellule 2 — Client Quiver minimal** (dans le notebook, pas de dépendance opaque). Endpoint bulk congresstrading V2.

In [ ]:
BASE = "https://api.quiverquant.com"
ENDPOINT = "/beta/bulk/congresstrading"
def quiver_page(page, page_size=500, version="V2"):
    headers = {"Authorization": f"Bearer {TOKEN}"}
    params = {"version": version, "normalized": "true", "nonstock": "false",
              "page": page, "page_size": page_size}
    r = requests.get(BASE + ENDPOINT, headers=headers, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    return data if isinstance(data, list) else data.get("results", data.get("data", []))

**Cellule 3 — Récupération paginée** (réglez `MAX_PAGES`).

In [ ]:
MAX_PAGES = 45
recs = []
for p in range(1, MAX_PAGES + 1):
    page = quiver_page(p)
    if not page:
        break
    recs.extend(page)
(EXT_QUIVER / "quiver_congress.json").write_text(json.dumps(recs))
print("Records Quiver :", len(recs))

**Cellule 4 — Normalisation Quiver** (pour comparaison seulement).

In [ ]:
q = pd.DataFrame(recs)
def g(df, *n):
    for x in n:
        if x in df.columns:
            return df[x]
    return pd.Series([None] * len(df))
quiver = pd.DataFrame({
    "declarant_name":   g(q, "Name", "Representative"),
    "chamber":          g(q, "Chamber", "House"),
    "ticker":           g(q, "Ticker"),
    "transaction_date": pd.to_datetime(g(q, "Traded", "TransactionDate"), errors="coerce"),
    "disclosure_date":  pd.to_datetime(g(q, "Filed", "ReportDate"), errors="coerce"),
})
print(len(quiver), "lignes Quiver")

**Cellule 5 — Triangulation de couverture par an.** Un écart Quiver ≠ une erreur de notre côté.

In [ ]:
ours = pd.read_csv(PROCESSED / "congress_transactions.csv", parse_dates=["disclosure_date"])
def by_year(df):
    return df["disclosure_date"].dt.year.value_counts().sort_index()
cmp = pd.DataFrame({"notre_table": by_year(ours), "quiver": by_year(quiver)}).fillna(0).astype(int)
cmp

**Cellule 6 — Rapport de validation.**

In [ ]:
from datetime import datetime, timezone
lines = ["# Validation Quiver (vérification externe)", "",
         f"Généré : {datetime.now(timezone.utc).isoformat(timespec='seconds')}", "",
         f"- Notre table : {len(ours)} transactions, {ours['declarant_name'].nunique()} déclarants",
         f"- Quiver (échantillon) : {len(quiver)} transactions, {quiver['declarant_name'].nunique()} déclarants", "",
         "## Couverture par an", "```", cmp.to_string(), "```", "",
         "> Quiver est une vérification ; jamais réinjecté dans la table finale."]
(REPORTS / "validation_report.md").write_text("\n".join(lines) + "\n")
print("Rapport écrit.")

Validation faite ✅ — rapport final **`09_data_quality_report`**.